# Weight survey B — quota sampler

5 cấu hình, 4M token, 500 step, seed 42.


In [ ]:
import glob, json, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available(): raise SystemExit('no GPU')
p=torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9,1), 'GB')
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], capture_output=True)
wheels=sorted(Path('/kaggle/input').rglob('unbabel_comet-*.whl'))
# không cần comet ở bước train; chỉ cần transformers/peft có sẵn


In [ ]:
DATA = Path('/kaggle/input/tiki-weight-ab-processed')
if not (DATA/'eval_A.parquet').exists():
    hits=glob.glob('/kaggle/input/**/eval_A.parquet', recursive=True)
    assert hits, 'không thấy dataset A/B'
    DATA=Path(hits[0]).parent
CODE=sorted(Path('/kaggle/input').rglob('train_weight_quota_gpu.py'))
assert CODE, 'không thấy code'
MODEL=None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try: d=json.loads(cfg.read_text(encoding='utf-8'))
    except Exception: continue
    if (d.get('hidden_size'),d.get('intermediate_size'),d.get('num_hidden_layers'))==(2560,9728,36):
        MODEL=str(cfg.parent); break
print('DATA',DATA); print('CODE',CODE[0]); print('MODEL',MODEL)
assert MODEL, 'không thấy Qwen3-4B'


In [ ]:
cmd=[sys.executable, str(CODE[0]), '--dataset', 'B',
     '--data-root', str(DATA), '--out', '/kaggle/working/weight_' + 'B', '--model', MODEL, '--seed','42', '--tokens','4000000', '--steps','500']
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
out=Path('/kaggle/working/weight_B')
for p in sorted(out.glob('hyp_*.parquet')): print(p.name,p.stat().st_size)
for p in sorted(out.glob('adapter_*')): print(p.name)
